# 🎬 Bài 4B: Xử lý Video với Computer Vision & YOLO
**Môn học**: E1402 - Digital and Computer Vision

**Thời gian**: ~45 phút | **Yêu cầu**: Đã hoàn thành Bài 4

---
## 🎯 Mục tiêu
1. Hiểu video = chuỗi ảnh (frames)
2. Xử lý video frame-by-frame với OpenCV
3. Phát hiện chuyển động (Motion Detection)
4. YOLO trên video - đếm vật thể qua thời gian
5. Tạo video output với annotation

> 💡 Video = **ảnh×liên tục** → mọi kỹ thuật ảnh đều áp dụng được cho video!

---
## 📖 Video trong Computer Vision

### Video = Chuỗi ảnh (Frames)
```
Video 30fps, 10 giây = 300 ảnh

Frame 1    Frame 2    Frame 3    ...    Frame 300
┌──────┐   ┌──────┐   ┌──────┐          ┌──────┐
│  🚗  │   │  🚗 │   │   🚗│          │     🚗
│      │   │      │   │      │          │      │
└──────┘   └──────┘   └──────┘          └──────┘
  33ms       33ms       33ms
```

### Các khái niệm quan trọng:
| Thuật ngữ | Ý nghĩa |
|---|---|
| **FPS** | Frames Per Second - số ảnh/giây (thường 24-30) |
| **Resolution** | Kích thước mỗi frame (VD: 1920×1080) |
| **Codec** | Thuật toán nén video (H.264, VP9...) |
| **Frame** | 1 ảnh trong video |

### Xử lý video với OpenCV:
```python
cap = cv2.VideoCapture('video.mp4')   # Mở video
ret, frame = cap.read()               # Đọc 1 frame
# frame = ảnh numpy array → xử lý bình thường!
cap.release()                          # Giải phóng
```

## ⚙️ Cài đặt

In [ ]:
!pip install -q ultralytics

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from urllib.request import urlopen, Request
from collections import Counter
import time
try:
    from google.colab import files
    COLAB = True
except ImportError:
    COLAB = False

from IPython.display import HTML
from base64 import b64encode
import tempfile, os

def hien_thi(ds_anh, ds_ten, figsize=(16, 5)):
    fig, axes = plt.subplots(1, len(ds_anh), figsize=figsize)
    if len(ds_anh) == 1: axes = [axes]
    for ax, anh, ten in zip(axes, ds_anh, ds_ten):
        if len(anh.shape) == 2: ax.imshow(anh, cmap='gray')
        else: ax.imshow(cv2.cvtColor(anh, cv2.COLOR_BGR2RGB))
        ax.set_title(ten, fontsize=12, fontweight='bold')
        ax.axis('off')
    plt.tight_layout()
    plt.show()

print('✅ Sẵn sàng!')

---
## 🎞️ Phần 1: Tạo và xử lý video

Bắt đầu bằng việc **tạo video giả lập** để hiểu cách video hoạt động

In [ ]:
# Tạo video giả lập: quả bóng di chuyển
rong, cao, fps = 640, 480, 20
so_frame = 60  # 3 giây video

# File output
video_path = 'video_test.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(video_path, fourcc, fps, (rong, cao))

# Tạo từng frame với quả bóng di chuyển
x_pos, y_pos = 100, 240  # Vị trí ban đầu
vx, vy = 8, 3            # Vận tốc
radius = 30

for i in range(so_frame):
    # Tạo nền
    frame = np.ones((cao, rong, 3), dtype=np.uint8) * 30  # Nền tối

    # Di chuyển bóng (nảy tường)
    x_pos += vx
    y_pos += vy
    if x_pos <= radius or x_pos >= rong - radius: vx = -vx
    if y_pos <= radius or y_pos >= cao - radius: vy = -vy
    x_pos = np.clip(x_pos, radius, rong - radius)
    y_pos = np.clip(y_pos, radius, cao - radius)

    # Vẽ bóng + trail
    cv2.circle(frame, (int(x_pos), int(y_pos)), radius, (0, 120, 255), -1)
    cv2.circle(frame, (int(x_pos), int(y_pos)), radius, (0, 200, 255), 2)

    # Thêm info
    cv2.putText(frame, f'Frame {i+1}/{so_frame}', (10, 30),
                cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
    cv2.putText(frame, f'X={int(x_pos)} Y={int(y_pos)}', (10, 65),
                cv2.FONT_HERSHEY_SIMPLEX, 0.6, (200, 200, 200), 1)

    writer.write(frame)

writer.release()
file_size = os.path.getsize(video_path) / 1024
print(f'✅ Đã tạo video: {video_path}')
print(f'   {so_frame} frames | {fps} FPS | {so_frame/fps:.1f} giây | {file_size:.0f} KB')

### Đọc video & hiển thị một số frames

In [ ]:
# Đọc lại video và hiển thị vài frame
cap = cv2.VideoCapture(video_path)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_v = cap.get(cv2.CAP_PROP_FPS)
w_v = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h_v = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print(f'📹 Video info:')
print(f'   Kích thước: {w_v}×{h_v}')
print(f'   FPS: {fps_v}')
print(f'   Tổng frames: {total}')
print(f'   Thời lượng: {total/fps_v:.1f}s')

# Lấy frames ở các vị trí khác nhau
vi_tri = [0, total//4, total//2, 3*total//4, total-1]
fig, axes = plt.subplots(1, len(vi_tri), figsize=(20, 4))
for ax, pos in zip(axes, vi_tri):
    cap.set(cv2.CAP_PROP_POS_FRAMES, pos)
    ret, f = cap.read()
    if ret:
        ax.imshow(cv2.cvtColor(f, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Frame {pos+1}', fontsize=11, fontweight='bold')
    ax.axis('off')
cap.release()
plt.suptitle('🎞️ Các frame trong video', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 🏃 Phần 2: Phát hiện chuyển động (Motion Detection)

### Nguyên lý cực đơn giản:
```
Frame N:        Frame N+1:       Sai khác: (|N+1 - N|)
┌──────┐       ┌──────┐         ┌──────┐
│      │       │      │         │      │
│  ●   │       │   ●  │   →     │  ██  │  ← Vùng thay đổi = chuyển động!
│      │       │      │         │      │
└──────┘       └──────┘         └──────┘
```

**Thuật toán**:
1. `absdiff(frame_truoc, frame_sau)` → tìm pixel thay đổi
2. Ngưỡng hóa → tạo mask chuyển động
3. Contours → khoanh vùng đang chuyển động

In [ ]:
# Phát hiện chuyển động trên video test
cap = cv2.VideoCapture(video_path)
ret, frame_truoc = cap.read()
xam_truoc = cv2.cvtColor(frame_truoc, cv2.COLOR_BGR2GRAY)
xam_truoc = cv2.GaussianBlur(xam_truoc, (21, 21), 0)  # Blur giảm nhiễu

ds_frames = []
frame_idx = 0

while True:
    ret, frame = cap.read()
    if not ret: break
    frame_idx += 1

    xam = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    xam = cv2.GaussianBlur(xam, (21, 21), 0)

    # Tính sai khác giữa 2 frame liên tiếp
    delta = cv2.absdiff(xam_truoc, xam)
    _, thresh = cv2.threshold(delta, 25, 255, cv2.THRESH_BINARY)
    thresh = cv2.dilate(thresh, None, iterations=2)

    # Tìm contour vùng chuyển động
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    annotated = frame.copy()
    for cnt in contours:
        if cv2.contourArea(cnt) < 500: continue
        x, y, w, h = cv2.boundingRect(cnt)
        cv2.rectangle(annotated, (x,y), (x+w,y+h), (0,255,0), 2)
        cv2.putText(annotated, 'MOTION', (x, y-10),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

    # Lưu 1 số frame
    if frame_idx % 10 == 0:
        ds_frames.append((annotated.copy(), delta.copy(), frame_idx))

    xam_truoc = xam

cap.release()

# Hiển thị kết quả
if ds_frames:
    so_hien = min(4, len(ds_frames))
    fig, axes = plt.subplots(2, so_hien, figsize=(5*so_hien, 8))
    for i in range(so_hien):
        anh, delta, idx = ds_frames[i]
        axes[0][i].imshow(cv2.cvtColor(anh, cv2.COLOR_BGR2RGB))
        axes[0][i].set_title(f'Frame {idx}', fontsize=11, fontweight='bold')
        axes[0][i].axis('off')
        axes[1][i].imshow(delta, cmap='hot')
        axes[1][i].set_title(f'Sai khác', fontsize=11)
        axes[1][i].axis('off')
    plt.suptitle('🏃 Phát hiện chuyển động', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()
print('📌 Vùng xanh = chuyển động | Heatmap = mức độ thay đổi pixel')

---
## 🚀 Phần 3: YOLO trên Video

Kết hợp YOLO + Video = **phát hiện vật thể liên tục** trên từng frame!

### 3.1 Tải video mẫu từ internet

In [ ]:
# Tải video mẫu ngắn (từ Pexels - miễn phí)
video_url = 'https://www.pexels.com/download/video/856065/'
video_file = 'traffic.mp4'

try:
    req = Request(video_url, headers={'User-Agent': 'Mozilla/5.0'})
    resp = urlopen(req, timeout=30)
    with open(video_file, 'wb') as f:
        f.write(resp.read())
    cap = cv2.VideoCapture(video_file)
    if cap.isOpened():
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
        print(f'✅ Đã tải video: {total} frames')
        cap.release()
    else:
        print('⚠️ Video không mở được, dùng video tự tạo')
        video_file = video_path
except:
    print('⚠️ Không tải được video online, dùng video tự tạo')
    video_file = video_path

### 3.2 YOLO xử lý video

In [ ]:
from ultralytics import YOLO

model = YOLO('yolov8n.pt')
cap = cv2.VideoCapture(video_file)
total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
fps_v = cap.get(cv2.CAP_PROP_FPS) or 20

# ===== CẤU HÌNH =====
so_frame_xu_ly = min(30, total)  # Xử lý bao nhiêu frame
# =====================

# Tạo video output với YOLO annotation
w = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
h = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
out_path = 'yolo_output.mp4'
writer = cv2.VideoWriter(out_path, cv2.VideoWriter_fourcc(*'mp4v'),
                          fps_v, (w, h))

ds_thong_ke = []  # Lưu thống kê mỗi frame
ds_preview = []   # Lưu frame để hiển thị

print(f'🎬 Xử lý {so_frame_xu_ly} frames...')
for i in range(so_frame_xu_ly):
    ret, frame = cap.read()
    if not ret: break

    t0 = time.time()
    results = model(frame, verbose=False)
    dt = (time.time() - t0) * 1000

    annotated = results[0].plot()
    writer.write(annotated)

    # Thống kê
    dem = Counter(model.names[int(b.cls[0])] for b in results[0].boxes)
    ds_thong_ke.append({'frame': i+1, 'objects': len(results[0].boxes),
                        'ms': dt, 'detail': dict(dem)})

    # Preview mỗi N frame
    if i % max(1, so_frame_xu_ly // 5) == 0:
        ds_preview.append((annotated.copy(), i+1, len(results[0].boxes)))

    if (i + 1) % 10 == 0:
        print(f'  Frame {i+1}/{so_frame_xu_ly}: {len(results[0].boxes)} vật thể ({dt:.0f}ms)')

cap.release()
writer.release()
print(f'\n✅ Đã lưu video output: {out_path}')

### 3.3 Kết quả xử lý

In [ ]:
# Hiển thị preview frames
if ds_preview:
    n = len(ds_preview)
    fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
    if n == 1: axes = [axes]
    for ax, (anh, idx, cnt) in zip(axes, ds_preview):
        ax.imshow(cv2.cvtColor(anh, cv2.COLOR_BGR2RGB))
        ax.set_title(f'Frame {idx}: {cnt} vật', fontsize=11, fontweight='bold')
        ax.axis('off')
    plt.suptitle('🎬 YOLO Video Processing', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

In [ ]:
# Đồ thị thống kê theo thời gian
if ds_thong_ke:
    frames = [s['frame'] for s in ds_thong_ke]
    objects = [s['objects'] for s in ds_thong_ke]
    times = [s['ms'] for s in ds_thong_ke]

    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 8))

    # Biểu đồ số vật thể
    ax1.fill_between(frames, objects, alpha=0.3, color='blue')
    ax1.plot(frames, objects, 'b-o', markersize=4)
    ax1.set_title('📦 Số vật thể phát hiện theo frame', fontsize=13, fontweight='bold')
    ax1.set_xlabel('Frame')
    ax1.set_ylabel('Số vật thể')
    ax1.grid(True, alpha=0.3)

    # Biểu đồ thời gian xử lý
    ax2.bar(frames, times, color='orange', alpha=0.7)
    ax2.axhline(y=np.mean(times), color='red', linestyle='--',
                label=f'Trung bình: {np.mean(times):.0f}ms')
    ax2.set_title('⏱️ Thời gian xử lý mỗi frame', fontsize=13, fontweight='bold')
    ax2.set_xlabel('Frame')
    ax2.set_ylabel('ms')
    ax2.legend()
    ax2.grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

    # Tổng kết
    print('📊 TỔNG KẾT:')
    print(f'   Frames xử lý: {len(ds_thong_ke)}')
    print(f'   Vật thể TB:   {np.mean(objects):.1f}/frame')
    print(f'   Thời gian TB: {np.mean(times):.0f}ms/frame')
    print(f'   FPS thực tế:  {1000/np.mean(times):.1f}')

    # Vật thể phổ biến nhất
    all_objects = Counter()
    for s in ds_thong_ke:
        all_objects.update(s['detail'])
    print('\n📦 Vật thể phổ biến:')
    for ten, so in all_objects.most_common(5):
        print(f'   {ten}: {so} lần xuất hiện')

---
## 📤 Phần 4: Xử lý video của bạn

Upload video hoặc nhập đường dẫn → YOLO xử lý từng frame

In [ ]:
# Upload hoặc nhập đường dẫn video
if COLAB:
    print('📤 Upload video (MP4 ngắn khuyến nghị)...')
    uploaded = files.upload()
    for ten, dl in uploaded.items():
        video_sv = ten
        with open(ten, 'wb') as f:
            f.write(dl)
        break
else:
    video_sv = input('📁 Nhập đường dẫn video (VD: /path/video.mp4): ')

cap = cv2.VideoCapture(video_sv)
if not cap.isOpened():
    print('❌ Không mở được video!')
else:
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    fps_v = cap.get(cv2.CAP_PROP_FPS)
    giay = total / fps_v if fps_v > 0 else 0
    print(f'📹 Video: {total} frames | {fps_v:.0f} FPS | {giay:.1f}s')

    # Xử lý tối đa 50 frame
    max_f = min(50, total)
    previews = []
    print(f'\n🎬 Xử lý {max_f} frames...')
    for i in range(max_f):
        ret, frame = cap.read()
        if not ret: break
        r = model(frame, verbose=False)
        if i % max(1, max_f//6) == 0:
            previews.append((r[0].plot(), i+1, len(r[0].boxes)))
            print(f'  Frame {i+1}: {len(r[0].boxes)} vật thể')
    cap.release()

    if previews:
        n = len(previews)
        fig, axes = plt.subplots(1, n, figsize=(5*n, 4))
        if n == 1: axes = [axes]
        for ax, (a, idx, c) in zip(axes, previews):
            ax.imshow(cv2.cvtColor(a, cv2.COLOR_BGR2RGB))
            ax.set_title(f'Frame {idx}: {c} vật', fontsize=11, fontweight='bold')
            ax.axis('off')
        plt.suptitle('🎬 YOLO trên video của bạn', fontsize=14, fontweight='bold')
        plt.tight_layout()
        plt.show()

---
## 📊 Tổng kết Bài 4B

### Kiến thức mới:
| Kỹ thuật | Hàm OpenCV | Ứng dụng |
|---|---|---|
| Đọc video | `cv2.VideoCapture()` | Mở camera/file |
| Ghi video | `cv2.VideoWriter()` | Lưu kết quả |
| Phát hiện chuyển động | `cv2.absdiff()` | An ninh, giám sát |
| YOLO trên video | `model(frame)` mỗi frame | Đếm xe, đếm người |

### Code chính:
```python
cap = cv2.VideoCapture('video.mp4')  # Mở video
while True:
    ret, frame = cap.read()          # Đọc frame
    if not ret: break
    results = model(frame)            # YOLO
    annotated = results[0].plot()     # Vẽ kết quả
    writer.write(annotated)           # Ghi video
cap.release()
```

### 🎉 Bạn đã nắm vững Computer Vision từ cơ bản đến nâng cao!

**Tiếp theo**: Huấn luyện YOLO với dữ liệu riêng, triển khai trên mobile, tích hợp camera an ninh 🚀